# Additional manoeuvre data: frozen NN pilots versus NN pilots

Acquire **10,000 new physical engagements (pairings)**, with **40,000 recorded aircraft perspectives**.
Both F-16s use the validation-selected, post-tactical-adaptation NN checkpoints reported
in [07_tactical_adaptation.ipynb](07_tactical_adaptation.ipynb). There are no PPO updates.
Every engagement has a fresh continuous relative geometry and seed. Flights stop at
**300 in-game seconds or the first aircraft defeat**, including a crash. Both pilots'
51-feature observations and physical tracks are recorded at **10 Hz**, and their actual
selected skills/parameters at **1 Hz**. All outcomes are retained.

**Run All downloads the verified pilot archive, benchmarks CPU and GPU-batched rollouts, collects the
campaign, verifies it, and commits/pushes incremental data backups to this branch with
Git LFS.** Every **Run All starts at zero** in a new series with a compact unique ID and new seed,
fresh setup/benchmark caches, verification receipts and progress logs. Earlier series
remain intact. Rerun only the collection cell to continue the series active in this
kernel. Set `COLLECT=False` and `PUBLISH=False` to inspect the setup first. A second
aircraft perspective never counts as another pairing.
The notebook is a long-running data acquisition job. Its benchmark estimates the actual
wall time and storage requirement before the main run. The original corpora remain usable.

Install the repository's dependencies in this kernel first: `%pip install -e ".[ml,video]"`.
Git and Git LFS must be installed, and `forecasting_ed_261104` must be checked out for
publication. Use `GIT_EXECUTABLE` below on Windows if Git is absent from the kernel PATH.
The companion script runs in fresh spawned processes on Windows and Linux.

**GPU update:** one inference process batches all ten independent actors across CPU simulator workers. Run All selects the measured fastest backend; CUDA needs a working CUDA-enabled PyTorch install. The full 20,000-flight series runs on your machine. Work/output roots default to `artifacts/nn_duels_20k_v1/` and `shared/corpora/nn-duels-20k-v1/`, with a separate subdirectory per Run All.

In [1]:
%%time
from pathlib import Path
import hashlib
import importlib.util
import importlib.metadata
import json
import os
import secrets
import shutil
import subprocess
import sys
import time

# PyCharm's project index and the running Jupyter kernel can use different roots.
root_override = os.getenv("BVR_NN_ROOT")
root_candidates = ([Path(root_override).expanduser()] if root_override else
                   [Path.cwd(), *Path.cwd().parents])
ROOT = next((p.resolve() for p in root_candidates
             if (p / "scripts/generate_nn_duel_corpus.py").is_file()), None)
if ROOT is None:
    raise RuntimeError(f"Cannot find the repository from kernel directory {Path.cwd()}. "
                       "Set BVR_NN_ROOT to the full Trajectory_Classification checkout path.")
sys.path.insert(0, str(ROOT / "scripts"))
importlib.invalidate_caches()

def load_companion(name):
    path = ROOT / "scripts" / f"{name}.py"
    if not path.is_file():
        raise FileNotFoundError(f"Missing companion script: {path}. Update the whole checkout.")
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    try:
        spec.loader.exec_module(module)
    except BaseException:
        sys.modules.pop(name, None)
        raise
    return module

# Every execution of this first cell starts a separate series; never reuse old state.
FLIGHTS = int(os.getenv("BVR_NN_FLIGHTS", "10000"))
SEED = int(os.getenv("BVR_NN_SEED", str(secrets.randbelow(2**31))))
EPISODE_SECONDS = float(os.getenv("BVR_NN_SECONDS", "300"))
SERIES_ID = secrets.token_hex(6)  # Keep Windows simulator paths below MAX_PATH.
WORK_ROOT = Path(os.getenv("BVR_NN_WORK", str(ROOT / "artifacts/nn_duels_20k_v1"))).resolve()
OUTPUT_ROOT = Path(os.getenv("BVR_NN_OUTPUT", str(ROOT / "shared/corpora/nn-duels-20k-v1"))).resolve()
WORK = WORK_ROOT / SERIES_ID
OUTPUT = OUTPUT_ROOT / SERIES_ID
WORK.mkdir(parents=True, exist_ok=False)
if OUTPUT.exists():
    raise FileExistsError(f"New series output already exists: {OUTPUT}")
print(f"Fresh series: {SERIES_ID} | seed: {SEED} | starting at 0/{FLIGHTS:,} pairings")

# Test the native library mix in a disposable process before importing it here.
native_runtime = load_companion("nn_duel_runtime")
native_runtime.run_preflight(WORK / "notebook_logs")
native_runtime.bootstrap_native_runtime()
import torch  # Torch must precede NumPy/Pandas/simulator imports on Windows.
import numpy as np
import pandas as pd
from IPython.display import display

duel = load_companion("generate_nn_duel_corpus")
NotebookProgress = load_companion("nn_duel_progress").NotebookProgress
duel_paths = load_companion("nn_duel_paths")
print(f"Kernel: {sys.executable} | repository: {ROOT}")

DOWNLOAD_LFS = os.getenv("BVR_NN_DOWNLOAD", "1") == "1"
RUN_BENCHMARK = os.getenv("BVR_NN_BENCHMARK", "1") == "1"
COLLECT = os.getenv("BVR_NN_COLLECT", "1") == "1"
PUBLISH = os.getenv("BVR_NN_PUBLISH", "1") == "1"
GIT_EXECUTABLE = None  # e.g. r"C:\Program Files\Git\cmd\git.exe"
DETERMINISTIC = True
FORCE_SPEED_COMMAND = False  # True only for the legacy forced +1 speed-request mode.
CUDA_AVAILABLE = torch.cuda.is_available()
BACKEND = os.getenv("BVR_NN_BACKEND", "auto")  # auto, central_cuda, local_cpu; central_cpu is for IPC tests.
BUDGET_WORKERS = max(1, min(32, int((os.cpu_count() or 1) * .8)))
WORKER_CANDIDATES = sorted({max(1, BUDGET_WORKERS//2), BUDGET_WORKERS})
BENCHMARK_FLIGHTS = int(os.getenv("BVR_NN_BENCHMARK_FLIGHTS", str(max(30, 2*BUDGET_WORKERS))))
WORKERS = BUDGET_WORKERS
BATCH_WAIT_MS = 2.0
BATCH_MAX_REQUESTS = 32
BACKUP_EVERY = 100
SHARD_FLIGHTS = 100
UPLOAD_MBPS = float(os.getenv("BVR_NN_UPLOAD_MBPS", "20"))  # Assumption; enter your sustained upload rate.
CHECKSUM_READ_MBPS = float(os.getenv("BVR_NN_SCAN_MBPS", "400"))  # Decimal MB/s, including cache effects.
GIT_SECONDS_PER_SNAPSHOT = 10.0
PROGRESS_INTERVAL_SECONDS = float(os.getenv("BVR_NN_PROGRESS_SECONDS", "300"))
FULLY_VERIFIED = False
SCRIPT = ROOT / "scripts/generate_nn_duel_corpus.py"
if BACKEND == "central_cuda" and not CUDA_AVAILABLE:
    raise RuntimeError("CUDA requested but unavailable. Install CUDA-enabled PyTorch or choose local_cpu/auto.")

runtime = {"logical_cpus": os.cpu_count(), "torch": str(torch.__version__),
           "torch_cuda_runtime": torch.version.cuda, "cuda_available": CUDA_AVAILABLE}
if CUDA_AVAILABLE:
    properties = torch.cuda.get_device_properties(0)
    runtime.update(gpu=properties.name, gpu_memory_gib=properties.total_memory/2**30,
                   compute_capability=str(torch.cuda.get_device_capability(0)))
    probe = torch.ones((16,16),device="cuda")
    probe = probe @ probe
    torch.cuda.synchronize()
    del probe
    runtime["cuda_kernel_check"] = "passed"
display(pd.Series(runtime, name="Detected execution hardware"))

updates = NotebookProgress(WORK, FLIGHTS, interval_seconds=PROGRESS_INTERVAL_SECONDS)

def run(command, *, work=WORK, output=OUTPUT, flights=FLIGHTS, seed=SEED,
        workers=None, backend=None, limit=None, full_verify=False):
    backend = backend or ("central_cuda" if BACKEND == "auto" and CUDA_AVAILABLE else
                          "local_cpu" if BACKEND == "auto" else BACKEND)
    args = [sys.executable, "-u", str(SCRIPT), command, "--work", str(work),
            "--output", str(output), "--flights", str(flights), "--seed", str(seed),
            "--seconds", str(EPISODE_SECONDS), "--workers", str(workers or WORKERS),
            "--shard-flights", str(SHARD_FLIGHTS), "--inference-backend", backend,
            "--batch-wait-ms", str(BATCH_WAIT_MS), "--batch-max-requests", str(BATCH_MAX_REQUESTS)]
    if limit is not None:
        args += ["--limit", str(limit)]
    if FORCE_SPEED_COMMAND:
        args += ["--force-speed-command"]
    if not DETERMINISTIC:
        args += ["--stochastic"]
    if command in ("verify", "publish"):
        args += ["--verification-cache", str(Path(work) / "verified_snapshot.json")]
    if command == "publish":
        args += ["--allow-partial"]
    env = os.environ.copy()
    if GIT_EXECUTABLE:
        env["PATH"] = str(Path(GIT_EXECUTABLE).parent) + os.pathsep + env.get("PATH", "")
    for key in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
        env[key] = "1"
    if full_verify:
        args = [sys.executable, "-u", str(ROOT / "scripts/nn_duel_progress.py"),
                "verify-complete", "--output", str(output)]
    stage = {"prepare": "Preparing frozen pilots and scenarios",
             "collect": "Collecting flights", "package": "Packaging saved flights",
             "verify": "Verifying saved data", "publish": "Verifying and uploading Git LFS backup"}[command]
    if command == "collect" and Path(work).resolve() != WORK:
        stage = f"Benchmark: {backend}, {workers or WORKERS} simulator workers"
    if full_verify:
        stage = "Final full verification"
    updates.run(args, cwd=ROOT, env=env, command=command, work=work, stage=stage, target=flights)

print(f"Target: {FLIGHTS:,} physical flights / {2*FLIGHTS:,} perspectives; <= {EPISODE_SECONDS}s")
print("Legacy speed-command override:", FORCE_SPEED_COMMAND)
print("Worker candidates:", WORKER_CANDIDATES, "| requested inference:", BACKEND)
print("Work:", WORK, "| LFS publication:", OUTPUT)
print(f"Human-readable progress about every {PROGRESS_INTERVAL_SECONDS/60:g} minutes; raw command logs: {updates.logs}")

Fresh series: eee3474c5493 | seed: 1883299364 | starting at 0/10,000 pairings
Checking native rollout libraries in a fresh process. Log: C:\Users\theon\Trajectory_Classification\artifacts\nn_duels_20k_v1\eee3474c5493\notebook_logs\native_runtime_ca803f92ddad.log
Native runtime preflight passed. Log: C:\Users\theon\Trajectory_Classification\artifacts\nn_duels_20k_v1\eee3474c5493\notebook_logs\native_runtime_ca803f92ddad.log
Kernel: C:\Users\theon\.conda\envs\trajectory-nn-clean\python.exe | repository: C:\Users\theon\Trajectory_Classification


logical_cpus                                          16
torch                                       2.14.1+cu130
torch_cuda_runtime                                  13.0
cuda_available                                      True
gpu                   NVIDIA GeForce RTX 5080 Laptop GPU
gpu_memory_gib                                 15.920471
compute_capability                               (12, 0)
cuda_kernel_check                                 passed
Name: Detected execution hardware, dtype: object

Target: 10,000 physical flights / 20,000 perspectives; <= 300.0s
Worker candidates: [6, 12] | requested inference: auto
Work: C:\Users\theon\Trajectory_Classification\artifacts\nn_duels_20k_v1\eee3474c5493 | LFS publication: C:\Users\theon\Trajectory_Classification\shared\corpora\nn-duels-20k-v1\eee3474c5493
Human-readable progress about every 5 minutes; raw command logs: C:\Users\theon\Trajectory_Classification\artifacts\nn_duels_20k_v1\eee3474c5493\notebook_logs
CPU times: total: 1.72 s
Wall time: 4.17 s


## Frozen population and actual initial conditions

The source is the **5.7 MB `reproduction.zip` in `trained-pilot-episodes-v1`**,
verified against its publication manifest. It contains all ten *post*-tactical
checkpoints, their architectures/configuration and the source/assets used for that
corpus. The notebook-05 public checkpoint pool predates tactical adaptation and is
not substituted. Each source file and checkpoint hash is checked before collection;
source, software and configuration must match for resumption.

The old spawn adapter used a sampled axis, opposing courses and fixed speeds; nominal
scenario bearings/headings were not actually applied. Here the requested position,
heading and speed are passed directly into aircraft creation during reset, so JSBSim
starts in that geometry. The collector checks actual reset positions and velocity
vectors. North/West/Up mathematical angles are converted into the proper velocity
vectors; the original guidance interface retains its own heading convention.

Each continuous dimension uses stratified samples. Ten families cover mixed aspects,
left/right crossings, tail chase, cold/recommit, close merges, low energy climbs,
high/low encounters, offset support and defensive beam geometries. Horizontal
separation spans 6–65 km, altitude 1.5–11.5 km, and speed 200–380 m/s.
Initial velocities exclude the old corpus's exactly antiparallel pattern; all
requested states are unique. Novel geometry is not a guarantee of novel NN behaviour.
The 100 ordered pilot pairs, including self-pairs, each receive 200 fresh flights
in the default campaign. Roles and pair order are balanced without reusing geometry.

Frozen package imports are pinned in each child process, including when this repository
is installed in editable mode. Failed benchmark candidates are retried on rerun.
Setup directories are keyed by collector sources, software and preparation settings.
Benchmark directories also include hardware and worker settings. Every Run All
allocates fresh WORK and OUTPUT subdirectories automatically, so no older campaign
configuration or source/software mismatch needs to be repaired to start this series.
The frozen collector still rejects incompatible changes within an active series.
Previously published data remains verifiable.


In [2]:
%%time
if DOWNLOAD_LFS or PUBLISH:
    # Retain the discovered executable so publication inherits the same Git PATH.
    sys.path.insert(0, str(ROOT))
    from bvr_behavior_prediction.paired_skill_forecasting import download_paired_corpora, resolve_git_runtime
    GIT_EXECUTABLE, git_environment = resolve_git_runtime(ROOT, GIT_EXECUTABLE)
    print("Git for download/publication:", GIT_EXECUTABLE)
    if PUBLISH:
        publication_branch = subprocess.run([GIT_EXECUTABLE, "branch", "--show-current"],
            cwd=ROOT, env=git_environment, check=True, text=True, capture_output=True).stdout.strip()
        if publication_branch != "forecasting_ed_261104":
            raise RuntimeError("Check out forecasting_ed_261104 before collection with publication enabled.")
        staged = subprocess.run([GIT_EXECUTABLE, "diff", "--cached", "--name-only"],
            cwd=ROOT, env=git_environment, check=True, text=True, capture_output=True).stdout.strip()
        if staged:
            raise RuntimeError("Commit or unstage existing staged changes before publication-enabled collection.")
        # Immutable corpus shards do not use LFS locks; the locking API may be unavailable.
        lfs_lock_key = "lfs.https://github.com/Rejn2024/Trajectory_Classification.git/info/lfs.locksverify"
        subprocess.run([GIT_EXECUTABLE, "config", "--local", lfs_lock_key, "false"],
            cwd=ROOT, env=git_environment, check=True)
        print("LFS lock verification disabled locally for the corpus endpoint; payload checks/uploads remain enabled.")
if DOWNLOAD_LFS:
    download_paired_corpora(ROOT, ["shared/corpora/trained-pilot-episodes-v1/reproduction.zip"],
                           git_executable=GIT_EXECUTABLE)
# Provision the frozen archive and geometry without fixing the main campaign backend yet.
setup_sources = {p.name: duel.sha256(p) for p in duel.collector_sources()}
setup_software = {p: importlib.metadata.version(p)
                  for p in ("numpy", "pandas", "pyarrow", "torch", "jsbsim", "gymnasium")}
setup_software["python"] = sys.version.split()[0]
# Temporary setup must vary with every input checked by prepare(), including a new kernel.
setup_identity = {"collector_hashes": setup_sources, "software": setup_software,
    "flights": FLIGHTS, "seed": SEED, "seconds": EPISODE_SECONDS,
    "deterministic": DETERMINISTIC, "native_throttle_override": FORCE_SPEED_COMMAND,
    "inference_backend": "local_cpu",
    "batch_wait_ms": BATCH_WAIT_MS, "batch_max_requests": BATCH_MAX_REQUESTS,
    "parent_archive_sha256": duel.sha256(duel.ARCHIVE),
    "parent_manifest_sha256": duel.sha256(duel.PARENT / "manifest.json")}
setup_id = hashlib.sha256(json.dumps(setup_identity, sort_keys=True).encode()).hexdigest()[:12]
SETUP_WORK = WORK / "setup" / setup_id
# Check every archive member for setup, the main run and the longest benchmark layout.
duel_paths.check_snapshot_paths(duel.ARCHIVE, [WORK, SETUP_WORK,
    WORK / "b" / ("0" * 12) / f"central_cuda_{max(WORKER_CANDIDATES)}"])
print("Preparing setup for this configuration:", SETUP_WORK)
run("prepare", work=SETUP_WORK, flights=FLIGHTS, seed=SEED, backend="local_cpu")
config = duel.read(SETUP_WORK / "config.json")
geometry = pd.DataFrame([{
    "family": s["family"], "pilot_A01": s["pilots"]["A01"], "pilot_B01": s["pilots"]["B01"],
    "horizontal_separation_m": np.linalg.norm(np.array(s["initial"]["B01"]["position"][:2]) -
                                             np.array(s["initial"]["A01"]["position"][:2])),
    "height_offset_m": s["initial"]["B01"]["position"][2] - s["initial"]["A01"]["position"][2],
    "heading_difference_deg": (s["initial"]["B01"]["heading_nwu_deg"] - s["initial"]["A01"]["heading_nwu_deg"]) % 360,
} for s in config["cases"]])
display(geometry.groupby("family").agg(flights=("family", "size"),
    minimum_range_m=("horizontal_separation_m", "min"), maximum_range_m=("horizontal_separation_m", "max"),
    minimum_height_offset_m=("height_offset_m", "min"), maximum_height_offset_m=("height_offset_m", "max")))
display(geometry.groupby(["pilot_A01", "pilot_B01"]).size().unstack(fill_value=0))
display(pd.DataFrame(config["pilots"]).T)

Git for download/publication: C:\Program Files\Git\cmd\git.exe
LFS lock verification disabled locally for the corpus endpoint; payload checks/uploads remain enabled.
Preparing setup for this configuration: C:\Users\theon\Trajectory_Classification\artifacts\nn_duels_20k_v1\eee3474c5493\setup\e15303a2abed


,flights,minimum_range_m,maximum_range_m,minimum_height_offset_m,maximum_height_offset_m
family,,,,,
close_merge,1000,6002.809632,13984.407963,-9827.346389,9380.398180
cold_recommit,1000,8046.592243,64824.160619,-9396.162944,9720.695652
crossing_left,1000,8023.941030,64897.322734,-9663.920987,9382.929282
crossing_right,1000,8009.086889,64994.978839,-9596.311607,9177.549109
defensive_beam,1000,10018.700213,24997.307083,-9325.549596,9446.335691
high_low,1000,8001.011939,64933.042222,-9909.086560,-3651.580155
low_energy_climb,1000,8226.707579,64988.363748,2006.679730,7777.148367
mixed_aspect,1000,8103.122974,64958.434274,-9534.335184,9381.568027
offset_support,1000,18025.181060,39993.205984,-9592.980543,9434.630977


pilot_B01,pilot_001,pilot_002,pilot_003,pilot_004,pilot_005,pilot_006,pilot_007,pilot_008,pilot_009,pilot_010
pilot_A01,,,,,,,,,,
pilot_001,100,100,100,100,100,100,100,100,100,100
pilot_002,100,100,100,100,100,100,100,100,100,100
pilot_003,100,100,100,100,100,100,100,100,100,100
pilot_004,100,100,100,100,100,100,100,100,100,100
pilot_005,100,100,100,100,100,100,100,100,100,100
pilot_006,100,100,100,100,100,100,100,100,100,100
pilot_007,100,100,100,100,100,100,100,100,100,100
pilot_008,100,100,100,100,100,100,100,100,100,100
pilot_009,100,100,100,100,100,100,100,100,100,100


,checkpoint_sha256,config_sha256,history_steps
pilot_001,dee45270d89a1df7f6cba65dee927266d28ed7c2490970...,f928fddb1e816a604e81730b51741ffde5313a7572a376...,20
pilot_002,9ae0761607c3e70c42bcfd4aa25909d084c06565246998...,9335df0b396835a0f49a18ca29c7073f8a6719e388ae56...,20
pilot_003,caeaefe2137e580379c3d01ec95ec65496513ae55d4da1...,47571ea284b48ec13b38b0d5a2c35b607b7a673a155d4c...,20
pilot_004,77c08f79a437b52fbd57d878d46bb05f129ef9e3656323...,061073a6edd99d10b380f8a5f7d78bd348c69b48bfedad...,20
pilot_005,5791be7ae3e90ec8f247d7370b4d006f6bd32b300db25f...,915338a5161df5ae7190908d2b48d71c7b467a6cbda0aa...,20
pilot_006,f13789f1a54ec9f06d94003dfcc886461cac8af6fce146...,6dc6e7097ad49a06a1adcd418d6783a7159edafdcc517e...,20
pilot_007,4835fd084c501372225cf1f44ce31bff3740611b718733...,4161d2ae21f17bdb0ee7b385c0286c9eeeaa21d8cc7e4c...,20
pilot_008,92aa526fd19a3ed7463244e5c749bf62f9f5b9b7dfc7de...,bf0a86e39a53231e7abd19ebca35788ffd7497ad4f2136...,20
pilot_009,c18033eff26c8ea96f4706176947fec5541ac0db01fc26...,28d0a85c1878f65ed5379778b68946f0e1e5fa8bdea5c4...,20
pilot_010,76ad0fbef45e0bfcf039919a90aced8f346bcf0e841f49...,3f0549b1835c005749836599e9ebab57e986f270264372...,20


CPU times: total: 188 ms
Wall time: 4.64 s


## Shared GPU actor batching and measured throughput

CPU processes retain independent JSBSim worlds, RNGs and two-second histories. With
`central_cuda`, **one additional process owns the CUDA context and all ten frozen
actors**. Workers send both aircraft histories over CPU pipes once per game second.
A configurable **2 ms maximum batching window** combines ready requests. `torch.func.vmap`
evaluates the independent actor weights together in one stacked forward; policies are
never averaged. Padding has no effect on returned actions. Only the actor encoder,
skill logits and parameter means run; unused critic and log-probability calculations
are omitted. The owner returns each selected skill and its original decoded parameters
in request order. Simulator workers never initialize CUDA or copy a model to the GPU.

This is deterministic FP32 inference with TF32 disabled. Every owner startup compares
80 histories across all ten actors against CPU reference logits/means and requires the
same selected skills, with `2e-5` numeric tolerances. This checks the actor math, not
bit-identical trajectories: GPU arithmetic and changing batch shapes can lead to small
differences that accumulate in combat. Source/precision/backend are recorded in the
immutable configuration. Changing backend requires a new campaign. Stochastic-policy
collection remains available with `local_cpu`; centralized batching does not silently
change stochastic sampling semantics.

A small cloud CPU profile put NN decisions at **12.5%** of rollout wall time: ideal
zero-cost inference would improve throughput by about **14%**. Most work is CPU physics,
sensors and missiles. GPU batching may therefore provide a modest gain or lose to IPC,
padding and launch costs. The benchmark compares **whole-flight throughput**, including
process/model startup, against local CPU inference, and selects the fastest successful
backend. Set `BACKEND="central_cuda"` to force the GPU option explicitly. CUDA failures
are shown in the results table; auto mode can retain a working CPU baseline.

The default suite has at least 30 full-duration-limited flights and at least two per
worker. Increase it to 100+ for a steadier estimate. All candidates use the same seeds,
pairs and starting geometry. Benchmark flights are separate from the main 20,000.
Unused rewards and scripted expert labels, videos and ACMI remain disabled. Physics
cadence and the native forced-throttle behaviour are unchanged. All prior payload bytes
are still checksummed before publication, while unchanged per-episode semantic checks
use a receipt keyed by the actual payload hashes, configuration and episode index row.
A final complete verification does not use that receipt.

In [3]:
%%time
benchmark_path = WORK / "benchmark_batched.json"
backends = (["local_cpu", "central_cuda"] if CUDA_AVAILABLE and DETERMINISTIC else ["local_cpu"]) if BACKEND == "auto" else [BACKEND]
benchmark_identity = {"collector_hashes": config["collector_hashes"], "software": config["software"],
    "runtime": runtime, "workers": WORKER_CANDIDATES, "backends": backends,
    "flights": BENCHMARK_FLIGHTS, "seconds": EPISODE_SECONDS, "deterministic": DETERMINISTIC,
    "batch_wait_ms": BATCH_WAIT_MS, "batch_max_requests": BATCH_MAX_REQUESTS,
    "native_throttle_override": FORCE_SPEED_COMMAND}
if RUN_BENCHMARK:
    cached = duel.read(benchmark_path) if benchmark_path.exists() else None
    # Reuse successful measurements, but retry failed candidates after a runtime repair.
    reusable = {(row["backend"], row["workers"]): row for row in cached["results"]
                if row["status"] == "ok"} if cached and cached["identity"] == benchmark_identity else {}
    rows = []
    bench_id = hashlib.sha256(json.dumps(benchmark_identity, sort_keys=True).encode()).hexdigest()[:12]
    for backend in backends:
        for workers in WORKER_CANDIDATES:
            if (backend, workers) in reusable:
                rows.append(reusable[(backend, workers)])
                continue
            bench_work = WORK / "b" / bench_id / f"{backend}_{workers}"
            try:
                run("collect", work=bench_work, flights=BENCHMARK_FLIGHTS, seed=SEED+100000,
                    workers=workers, backend=backend)
                summaries = [duel.completed(bench_work, s) for s in duel.read(bench_work / "config.json")["cases"]]
                progress = duel.read(bench_work / "progress.json")
                elapsed = progress.get("cumulative_wall_seconds", progress["wall_seconds"])
                signatures = sorted((s["episode_id"], sorted(s["files"].items())) for s in summaries)
                post_start = time.perf_counter()
                run("package", work=bench_work, output=bench_work / "published", flights=BENCHMARK_FLIGHTS,
                    seed=SEED+100000, backend=backend)
                run("verify", work=bench_work, output=bench_work / "published", backend=backend)
                post_seconds = time.perf_counter()-post_start
                inference = duel.read(bench_work / "inference_metrics.json") if (bench_work / "inference_metrics.json").exists() else {}
                rows.append({"backend": backend, "workers": workers, "status": "ok",
                    "wall_seconds": elapsed, "flights_per_hour": BENCHMARK_FLIGHTS*3600/elapsed,
                    "postprocess_seconds": post_seconds,
                    "mean_game_seconds": np.mean([s["duration_s"] for s in summaries]),
                    "decision_fraction": sum(s["timing"]["decision_seconds"] for s in summaries)/sum(s["wall_seconds"] for s in summaries),
                    "bytes_per_flight": sum(p.stat().st_size for p in (bench_work / "episodes").rglob("*.parquet"))/BENCHMARK_FLIGHTS,
                    "mean_actor_rows_per_gpu_dispatch": inference.get("mean_rows_per_dispatch", 0),
                    "data_signature": hashlib.sha256(json.dumps(signatures).encode()).hexdigest()})
            except subprocess.CalledProcessError as error:
                rows.append({"backend": backend, "workers": workers, "status": "failed",
                    "error": f"{error}\n{error.output or error.stderr or ''}"})
    benchmark = pd.DataFrame(rows)
    cpu = benchmark[(benchmark.backend == "local_cpu") & (benchmark.status == "ok")]
    if DETERMINISTIC and len(cpu) and cpu.data_signature.nunique() != 1:
        raise RuntimeError("CPU worker counts produced different deterministic data; investigate before collection")
    duel.write(benchmark_path, {"identity": benchmark_identity, "results": rows})
    display(benchmark.drop(columns="data_signature", errors="ignore"))
    successful = benchmark.loc[benchmark.status.eq("ok")].copy()
    if successful.empty:
        failures = "\n\n".join(f"{row.backend}, {row.workers} workers:\n{row.error}"
                                 for row in benchmark.itertuples())
        raise RuntimeError("Every rollout backend failed. Command diagnostics:\n" + failures)
    best = successful.loc[successful.flights_per_hour.idxmax()]
    BACKEND, WORKERS = str(best.backend), int(best.workers)
    # An existing campaign must keep its recorded inference backend for reproducibility.
    if (WORK / "config.json").exists():
        pinned_backend = duel.read(WORK / "config.json")["inference_backend"]
        if BACKEND != pinned_backend:
            matches = successful.loc[successful.backend.eq(pinned_backend)]
            if matches.empty:
                raise RuntimeError("Existing campaign backend failed its benchmark; use a new campaign to change backend.")
            best = matches.loc[matches.flights_per_hour.idxmax()]
            BACKEND, WORKERS = str(best.backend), int(best.workers)
            print("Resumption retains the existing campaign backend:", BACKEND)
    estimate = duel.estimate_campaign(FLIGHTS, BENCHMARK_FLIGHTS, float(best.wall_seconds),
        float(best.postprocess_seconds), float(best.bytes_per_flight), SHARD_FLIGHTS,
        UPLOAD_MBPS, CHECKSUM_READ_MBPS, GIT_SECONDS_PER_SNAPSHOT)
    display(pd.Series(estimate, name="Estimated full campaign, including LFS upload"))
    print(f"Selected {BACKEND}, {WORKERS} simulator workers.")
    print(f"Planning estimate: {estimate['total_hours']:.1f} hours; sensitivity band "
          f"{estimate['sensitivity_low_hours']:.1f}–{estimate['sensitivity_high_hours']:.1f} hours, "
          f"plus the one-off {len(benchmark)}-candidate benchmark.")
    print(f"Upload assumption {UPLOAD_MBPS:g} Mbps; checksum-read assumption {CHECKSUM_READ_MBPS:g} MB/s. Edit these for your machine.")
    duel.write(WORK / "runtime_estimate.json", {"backend": BACKEND, "workers": WORKERS, **estimate})
    free_gib = shutil.disk_usage(WORK).free/2**30
    if free_gib < 3.5*estimate["payload_gib"]:
        raise RuntimeError("Insufficient free disk for local work, publication and LFS object storage")
else:
    BACKEND = ("central_cuda" if CUDA_AVAILABLE and DETERMINISTIC else "local_cpu") if BACKEND == "auto" else BACKEND
    print("Benchmark disabled; explicitly selected", BACKEND, WORKERS)
run("prepare", backend=BACKEND)
config = duel.read(WORK / "config.json")

,backend,workers,status,wall_seconds,flights_per_hour,postprocess_seconds,mean_game_seconds,decision_fraction,bytes_per_flight,mean_actor_rows_per_gpu_dispatch
0,local_cpu,6,ok,30.301869,3564.136553,7.631430,236.31,0.102191,1207854.5,0.000000
1,local_cpu,12,ok,27.683127,3901.293390,7.564341,236.31,0.087535,1207854.5,0.000000
2,central_cuda,6,ok,47.079948,2293.970258,6.840347,236.31,0.310759,1207846.3,4.554093
3,central_cuda,12,ok,36.031202,2997.402077,6.877872,236.31,0.268971,1207845.4,6.039166


rollout_hours                          2.563252
package_and_semantic_verify_hours      0.700402
repeated_checksum_hours                1.660800
upload_hours                           1.342061
git_overhead_hours                     0.277778
total_hours                            6.544293
sensitivity_low_hours                  4.897979
sensitivity_high_hours                 9.836920
payload_gib                           11.249022
assumed_upload_mbps                   20.000000
assumed_checksum_read_MBps           400.000000
assumed_git_seconds_per_snapshot      10.000000
Name: Estimated full campaign, including LFS upload, dtype: float64

Selected local_cpu, 12 simulator workers.
Planning estimate: 6.5 hours; sensitivity band 4.9–9.8 hours, plus the one-off 4-candidate benchmark.
Upload assumption 20 Mbps; checksum-read assumption 400 MB/s. Edit these for your machine.
CPU times: total: 734 ms
Wall time: 3min 10s


## How long should the complete process take?

The benchmark estimates the 20,000-pairing series for your machine, including
checksums, packaging and LFS uploads, assuming initially **20 Mbps sustained upload**.
The earlier 10,000-flight estimates do not cover this larger series. The old cloud benchmark measured
528 flights/hour, or about **37.9 hours for 20,000 rollouts alone on that environment**.
A simple eight-effective-worker extrapolation from its 80.75-second serial ten-flight
suite gives about **5.6 hours for acquisition** before storage/network overhead. GPU
offload may reduce that modestly; it cannot accelerate CPU JSBSim physics.

The benchmark above replaces this extrapolation with your machine's measured whole-flight
rate and measured packaging/verification time. It separately budgets repeated prefix
checksum reads, fixed Git overhead and dataset upload at the entered rate. These disk,
network and scaling assumptions are exposed; the sensitivity range is not a statistical
confidence interval. At the measured corpus density, expect roughly **20–30 GiB** of
compressed payload and allow at least **80–110 GiB free** for work, publication and LFS
objects. Every 100-flight cycle then updates an ETA using actual elapsed collection,
verification and upload time. CUDA execution cannot be validated on this CPU-only
cloud environment; the owner's startup parity test and the on-machine benchmark are
required checks before accepting a GPU speedup.

## Collection, restart and off-machine backups

Completed episodes have six checksummed Parquet files (three per perspective) and
an atomic, fsynced completion marker. An interrupted flight is rerun with the same
seed; successful flights are verified and skipped. A campaign lock prevents two
kernels writing the same work directory. After a hard crash, inspect the PID in
`collection.lock` and remove that file only once the process has stopped.

Every 100 new flights the notebook closes immutable shards, verifies published data,
and, when `PUBLISH=True`, commits **only the corpus directory** and pushes this branch.
Git's pre-push hook uploads LFS payloads; publication checks the remote commit afterward.
Partial backups have `complete=false` and explicit physical-flight counts. The final
manifest can say `complete=true` only at 20,000 flights by default. A failed push raises immediately;
local episode checkpoints remain available to rerun the cell. Unrelated staged changes
are rejected by publication. Existing committed branch work will accompany a push.

Publication setup disables Git LFS lock verification **locally for this repository's
GitHub LFS endpoint**: immutable corpus shards do not use file locks. This avoids
`Fatal error: Unable to verify locks` when the lock API is unavailable. Payload
verification, the LFS pre-push upload hook and remote-commit confirmation still run.

Every execution of the first cell allocates a new `SERIES_ID`, WORK and OUTPUT.
`BVR_NN_WORK` and `BVR_NN_OUTPUT` now select **parent directories**, including if
they still name an older campaign: this series always uses its own subdirectory.
The default seed is freshly chosen and printed; set `BVR_NN_SEED` only when you
intentionally want the same deterministic scenario schedule in a separate series.
`BVR_NN_FLIGHTS` can override 20,000 for a shorter validation run.

Series IDs are compact and benchmarks use `b/` to keep extracted simulator paths
within Windows limits. Setup checks every archive destination before extraction.
If an overridden work root is still too long, the error explains how to select
a short `BVR_NN_WORK` parent such as `C:/bvrw`. Output can remain in the repository.

To start over, use Run All. To continue an interrupted series in the same kernel,
rerun **only the collection cell**. To retry a failed backup before collecting
more flights, use `run("publish", backend=BACKEND)`, then rerun the collection cell.
Keep the current WORK and OUTPUT for that retry. Executing the first cell instead
starts another series. Earlier directories and published corpora are preserved.

Keep the work directory until the final package is verified and its upload is confirmed.
LFS storage and network transfer grow with the measured corpus size. Publication uses
bounded shards below 1 GB, not a single giant file; it does not rewrite old corpora.

The selected CPU/GPU backend is fixed in the campaign configuration. The inference owner is restarted once per backup chunk; its startup is included in the benchmark. Publication verifies each snapshot once and reuses semantic receipts for unchanged episodes. Final verification rereads the entire corpus without a receipt. Live cycle timing updates the remaining-time estimate after packaging and uploads.

### Five-minute progress updates

The notebook prints a short human-readable status about every **300 wall-clock
seconds**, configurable with `PROGRESS_INTERVAL_SECONDS` in the first cell. It
shows saved physical flights and aircraft perspectives, percent complete, elapsed
time, observed flights/hour and ETA, current stage, CPU/GPU backend and worker
count, failed flights in the latest batch, and the most recent verified/LFS-pushed
backup. Benchmark progress is labelled separately. Start, final completion and
errors are reported immediately; backup boundaries do not flood the output.

The same clock spans collection chunks, packaging, verification and LFS uploads,
so long non-rollout stages also get a keepalive. Raw JSON, simulator warnings and
Git output go directly to unique local files in `WORK/notebook_logs/`; the human
updates have their own transcript there. Failure output includes a short log tail.
When a command has not yet completed a flight, the update says so instead of
claiming a stall or inventing an ETA.

Reporting stays in the notebook process: one blocked waiter thread waits for a
command to exit, and a timer wakes the notebook roughly once per five minutes.
Only the existing small `progress.json` counter is read at those intervals.
There are no added worker messages, trajectory scans, checksums, GPU utilization
queries or CUDA synchronizations for reporting. The flight collector and its
source hashes are unchanged, so this reporting update preserves existing campaign
resumption. Final full verification runs as an observed subprocess without a cache;
completion is printed only after that verification succeeds.


In [4]:
%%time
if COLLECT:
    cycle_start = time.perf_counter()
    start_count = sum(duel.completed(WORK, s, verify=False) is not None for s in config["cases"])
    updates.start_collection(start_count, BACKEND, WORKERS)
    while True:
        config = duel.read(WORK / "config.json")
        count = sum(duel.completed(WORK, s, verify=False) is not None for s in config["cases"])
        if count < FLIGHTS:
            run("collect", workers=WORKERS, backend=BACKEND, limit=min(BACKUP_EVERY, FLIGHTS-count))
        run("package", backend=BACKEND)
        if PUBLISH:
            run("publish", backend=BACKEND)  # Includes checksum and semantic verification.
        else:
            run("verify", backend=BACKEND)
        manifest = duel.read(OUTPUT / "manifest.json")
        completed = manifest["physical_flights"]
        elapsed = time.perf_counter()-cycle_start
        acquired = completed-start_count
        remaining_hours = (FLIGHTS-completed)*elapsed/max(1, acquired)/3600
        updates.snapshot(completed, pushed=PUBLISH)
        duel.write(WORK / "live_eta.json", {"completed": completed, "elapsed_seconds": elapsed,
                                           "estimated_remaining_hours": remaining_hours})
        if manifest["complete"]:
            break
    run("verify", full_verify=True)
    FULLY_VERIFIED = True
    updates.finish()
else:
    run("status")
    print("Collection disabled; this execution did not start the main campaign.")

[2026-11-08 00:12:09 UTC] Starting collection: 0/10,000 flights already saved; local_cpu, 12 simulator workers. Updates about every 5m 00s; detailed command logs: C:\Users\theon\Trajectory_Classification\artifacts\nn_duels_20k_v1\eee3474c5493\notebook_logs
[2026-11-08 00:17:09 UTC] Collecting flights
Flights saved: 283/10,000 (2.8%); 566 aircraft perspectives; collection elapsed 5m 00s.
Observed rate: 3,396 flights/hour; estimated remaining 2h 51m including observed backup time.
Backend: CPU inference; configured simulator workers: 12; failed flights in latest collection batch: 0.
Last backup: 200 flights verified and pushed to Git LFS, 1m 09s ago.
Latest completed-flight counter updated 0s ago.
[2026-11-08 00:22:09 UTC] Collecting flights
Flights saved: 519/10,000 (5.2%); 1,038 aircraft perspectives; collection elapsed 10m 00s.
Observed rate: 3,114 flights/hour; estimated remaining 3h 02m including observed backup time.
Backend: CPU inference; configured simulator workers: 12; failed 

## Data contract for a later 09b-style experiment

`A01/` stores A01's observation of NN target B01; `B01/` stores B01's observation of
NN target A01. Each has `observations`, `tracks` and privileged `target_decisions`.
`episodes.parquet` identifies target/observer pilots and checkpoint hashes, actual
initial state, shared `scenario_group`, duration and outcome. Parquet row groups are
whole episodes. Time zero and the terminal sample are retained. A terminal sample
has no newly issued decision; future labels are never extended past the final action.

The helper below returns one perspective in the same column contract used by `09b`.
For the A01-observer direction only, its copy of the decision ID is canonicalized to
`A01`, the target name required by `09b.forecast_targets`; `actual_aircraft_id` preserves
physical identity. Observation axes and values are unchanged. This enables the same
**54 causal features**, **51-frame histories plus five acceleration lookback frames**,
first origin at **6 s**, and exact **+1/+2/+5 s** targets with independent missing masks.
The existing `09b.prepare_dataset` hardcodes its old corpus; it does not automatically
consume this new release. A later preparation pass must index these shards explicitly.

Keep **both views of each engagement in the same split**. Fit normalization only on
training histories. Decisions, pilot/checkpoint identity, future tracks and outcomes
remain outside encoder inputs. Combining two observations into 108 inputs is a
separate omniscient task unless both streams are available at inference time.
Reserve validation before using skill-gap feedback to tune scenario distributions;
actively selected follow-up data belongs in training, with a separate fixed evaluation
suite. Catalogue labels can share executed control code and are not physical-skill proof.

In [5]:
%%time
if (OUTPUT / "manifest.json").exists():
    if not FULLY_VERIFIED:
        run("verify")
    manifest = duel.read(OUTPUT / "manifest.json")
    episodes = pd.read_parquet(OUTPUT / "episodes.parquet")
    display(episodes.groupby(["observer_id", "family"]).agg(
        perspectives=("episode_id", "size"), mean_duration_s=("duration_s", "mean")))
    if len(episodes):
        episode = episodes.iloc[0]
        observations, tracks, target_decisions = duel.load_forecasting_view(
            OUTPUT, episode.episode_id, episode.observer_id)
        from bvr_behavior_prediction.paired_skill_forecasting import trained_examples
        classes = duel.read(WORK / "tactical_snapshot/training/pilot_001/config.json")["skill_names"]
        features, origins, targets = trained_examples(observations, tracks, target_decisions, classes)
        display(pd.Series({"physical_flights": manifest["physical_flights"],
            "perspectives": manifest["perspective_episodes"], "example_feature_shape": features.shape,
            "forecast_origins": len(origins), "valid_labels_per_horizon": (targets >= 0).sum(axis=0).tolist()}))
        display(target_decisions.head())
else:
    print("No collection snapshot yet.")

perspectives  mean_duration_s
observer_id family                                         
A01         close_merge               1000         110.4747
            cold_recommit             1000         277.7224
            crossing_left             1000         237.1058
            crossing_right            1000         214.5859
            defensive_beam            1000         103.1956
            high_low                  1000         246.7570
            low_energy_climb          1000         261.1450
            mixed_aspect              1000         233.6012
            offset_support            1000         183.1533
            tail_chase                1000         241.0621
B01         close_merge               1000         110.4747
            cold_recommit             1000         277.7224
            crossing_left             1000         237.1058
            crossing_right            1000         214.5859
            defensive_beam            1000         103.1956
            high_low                  1000         246.7570
            low_energy_climb          1000         261.1450
            mixed_aspect              1000         233.6012
            offset_support            1000         183.1533
            tail_chase                1000         241.0621

physical_flights                      10000
perspectives                          20000
example_feature_shape            (3001, 54)
forecast_origins                        293
valid_labels_per_horizon    [293, 292, 289]
dtype: object

,episode_id,aircraft_id,step,time_s,skill,parameters_json,actual_aircraft_id
0,nn_1883299364_00000,A01,0,0.0,secondary_shot,"{""launch_range_m"": 105448.7578125, ""short_rang...",B01
1,nn_1883299364_00000,A01,10,1.0,lag_pursuit,"{""capture_angle_deg"": 42.624454498291016, ""com...",B01
2,nn_1883299364_00000,A01,20,2.0,turn_cold,"{""capture_angle_deg"": 42.292259216308594, ""com...",B01
3,nn_1883299364_00000,A01,30,3.0,turn_cold,"{""capture_angle_deg"": 42.192264556884766, ""com...",B01
4,nn_1883299364_00000,A01,40,4.0,turn_cold,"{""capture_angle_deg"": 42.1838264465332, ""commi...",B01


CPU times: total: 93.8 ms
Wall time: 131 ms


## Where might the missing skills appear? Hypotheses to test

The saved pre-split survey in `09b` on this branch examined 3,000 NN-versus-scripted
flights. **`commit`, `dive_defense`, and `drag_missile` had zero eligible episodes at
all three forecast horizons.** `accelerate_to_speed` and `climb_to_altitude` had only
**two episodes** each; `lock_target` had four. These are archived observations, not a
promise that a larger corpus will make those classes appear. The new audit below
uses the same six-second eligibility and exact-horizon rules and retains zero classes.

* **Energy/altitude gaps:** low-energy starts with a higher opponent may encourage
  acceleration and climbing; inverted height advantages may encourage descent and
  vertical defence. The low-energy/climb and high/low families test this. New series
  honour requested speed deltas (`FORCE_SPEED_COMMAND=False`). Set this to True only
  to request the legacy forced +1 mode. This is a speed-request override, not a fixed
  engine-throttle position. The mode is recorded in config and cache identities; use
  a new series when changing it. Frozen pilots were trained with the override, so
  reassess their behaviour in the new mode. Catalogue labels still need separate
  physical-behaviour validation; disabling the override does not implement missing
  manoeuvre controllers.
* **Commit/recommit:** cold, crossing and tail-chase starts place targets behind or
  outside the initial radar cone and may create search, lock and re-engagement
  transitions. Offset/support starts vary missile support geometry. Prolonging flights
  allows more post-shot/recommit phases, but first defeat still terminates the episode.
* **Missile defence gaps:** close merges and defensive beam starts may elicit earlier
  opposing launches and change time-to-impact and aspect, possibly yielding dive,
  drag or last-ditch responses. Recording incoming missiles and remaining weapons
  helps distinguish lack of threat exposure from refusal to select a defensive skill.
* **Adaptive next batch (speculation, not implemented):** after an initial development
  batch, relate eligible skill counts to family, energy gap, aspect, range, pilot pair
  and missile threat exposure. Sample undercovered cells and perturb promising starts;
  use Bayesian optimization or a cross-entropy search over initial conditions with
  rare-skill episode support as its score. Keep a broad random component, log sampling
  probabilities and seeds, and avoid success-only filtering. Count independent scenario
  groups and transitions, not just thousands of overlapping windows.
* **Policy support limits:** a frozen deterministic pilot may never choose a catalogue
  entry anywhere reachable. Inspect its logits on diverse histories first. A separately
  labelled stochastic-policy campaign (`DETERMINISTIC=False`, fresh work/output) can
  sample lower-probability choices while preserving NN control; it changes the behaviour
  distribution. If the missing skills have essentially zero probability, use curriculum
  fine-tuning or skill-conditioned demonstrators in a separate study. Forced catalogue
  actions would not demonstrate that these frozen pilots independently chose the skill.

Scenario novelty, catalogue coverage, physical behaviour and forecast learnability
are four distinct measurements. Report both sides and pilot-specific counts before
claiming a gap has been filled; a zero may persist despite all 20,000 new flights.

In [6]:
%%time
if (OUTPUT / "skill_support.parquet").exists():
    support = pd.read_parquet(OUTPUT / "skill_support.parquet")
    display(support.pivot(index="skill", columns="horizon_s", values="windows"))
    display(support.pivot(index="skill", columns="horizon_s", values="perspective_episodes"))
    print("Still missing at one or more horizons:")
    display(support[support.windows.eq(0)])
    breakdown = pd.read_parquet(OUTPUT / "skill_support_by_pilot_family.parquet")
    display(breakdown.groupby(["pilot_id", "horizon_s"]).agg(supported_skills=("skill", lambda s: s[breakdown.loc[s.index, "windows"].gt(0)].nunique())))
    print("These episode counts include two correlated perspectives per physical engagement; split by scenario_group.")
else:
    print("Skill support will be measured from actual selections after the first closed shard.")

horizon_s,1,2,5
skill,,,
abort_support,4970,4929,4740
accelerate_to_speed,6,6,6
beam_missile_left,846,827,734
beam_missile_right,3587,3390,2788
beam_target_left,163502,161791,156914
beam_target_right,75138,73520,69027
climb_to_altitude,64,63,57
commit,7,7,4
crank_target_left,57298,54016,45000


horizon_s,1,2,5
skill,,,
abort_support,379,378,370
accelerate_to_speed,2,2,2
beam_missile_left,149,142,130
beam_missile_right,478,471,444
beam_target_left,5981,5853,5487
beam_target_right,4310,4228,4021
climb_to_altitude,17,17,16
commit,2,2,2
crank_target_left,5806,5596,4774


Still missing at one or more horizons:


,horizon_s,skill,windows,perspective_episodes


supported_skills
pilot_id  horizon_s                  
pilot_001 1                        24
          2                        24
          5                        24
pilot_002 1                        27
          2                        26
          5                        26
pilot_003 1                        28
          2                        27
          5                        25
pilot_004 1                        27
          2                        27
          5                        27
pilot_005 1                        26
          2                        26
          5                        26
pilot_006 1                        24
          2                        24
          5                        24
pilot_007 1                        28
          2                        28
          5                        28
pilot_008 1                        26
          2                        26
          5                        26
pilot_009 1                        27
          2                        27
          5                        27
pilot_010 1                        25
          2                        25
          5                        25

These episode counts include two correlated perspectives per physical engagement; split by scenario_group.
CPU times: total: 46.9 ms
Wall time: 57.2 ms
